## ICC World Cup — Points Table

### Table: `icc_world_cup`

| Column Name | Type |
| --- | --- |
| `team_1` | `varchar(20)` |
| `team_2` | `varchar(20)` |
| `winner` | `varchar(20)` |

* Each record represents a match between two teams and the winner of that match.

### Task
Write a SQL query to generate a points table that shows for each team:

* Matches played
* Wins
* Losses
* Points (2 points per win)

### Example

**Input:** `icc_world_cup`

| team_1 | team_2 | winner |
| --- | --- | --- |
| India | Sri Lanka | India |
| India | Australia | India |
| India | England | England |
| India | New Zealand | India |
| Australia | England | Australia |
| Australia | New Zealand | New Zealand |
| Australia | South Africa | South Africa |
| Australia | Sri Lanka | Australia |
| England | New Zealand | New Zealand |
| England | South Africa | England |
| England | Sri Lanka | England |
| England | Pakistan | Pakistan |
| New Zealand | South Africa | South Africa |
| New Zealand | Pakistan | New Zealand |
| New Zealand | Sri Lanka | New Zealand |
| South Africa | Pakistan | Pakistan |
| South Africa | Sri Lanka | South Africa |
| Pakistan | Sri Lanka | Pakistan |
| Pakistan | India | India |
| Bangladesh | Sri Lanka | Sri Lanka |
| Bangladesh | Pakistan | Pakistan |
| Bangladesh | India | India |
| Bangladesh | England | England |

**Output:**

| team_name | no_of_matches_played | no_of_matches_won | no_of_losses |
| --- | --- | --- | --- |
| India | 4 | 3 | 1 |
| Australia | 4 | 2 | 2 |
| England | 4 | 2 | 2 |
| New Zealand | 3 | 2 | 1 |
| South Africa | 2 | 1 | 1 |
| Pakistan | 2 | 1 | 1 |
| Bangladesh | 4 | 0 | 4 |

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("PointsTable").getOrCreate()

data = [
        ("India", "Sri Lanka", "India"),
        ("India", "Australia", "India"),
        ("India", "England", "England"),
        ("India", "New Zealand", "India"),
        ("Australia", "England", "Australia"),
        ("Australia", "New Zealand", "New Zealand"),
        ("Australia", "South Africa", "South Africa"),
        ("Australia", "Sri Lanka", "Australia"),
        ("England", "New Zealand", "New Zealand"),
        ("England", "South Africa", "England"),
        ("England", "Sri Lanka", "England"),
        ("England", "Pakistan", "Pakistan"),
        ("New Zealand", "South Africa", "South Africa"),
        ("New Zealand", "Pakistan", "New Zealand"),
        ("New Zealand", "Sri Lanka", "New Zealand"),
        ("South Africa", "Pakistan", "Pakistan"),
        ("South Africa", "Sri Lanka", "South Africa"),
        ("Pakistan", "Sri Lanka", "Pakistan"),
        ("Pakistan", "India", "India"),
        ("Bangladesh", "Sri Lanka", "Sri Lanka"),
        ("Bangladesh", "Pakistan", "Pakistan"),
        ("Bangladesh", "India", "India"),
        ("Bangladesh", "England", "England")
]

icc_world_cup = spark.createDataFrame(data, schema = ["team_1", "team_2", "winner"])

icc_world_cup.createOrReplaceTempView("icc_world_cup")

In [0]:
%sql
SELECT
    team_1 AS team_name
    , COUNT(*) AS no_of_matches_played
    , SUM(CASE WHEN team_1 = winner THEN 1 ELSE 0 END) AS no_of_matches_won
    , SUM(CASE WHEN team_1 != winner THEN 1 ELSE 0 END) AS no_of_losses
FROM
    icc_world_cup
GROUP BY
    team_1
ORDER BY
    no_of_matches_won DESC;

In [0]:
from pyspark.sql.functions import col, when, sum, count

result = (icc_world_cup
            .groupBy(col("team_1"))
            .agg(
                count("*").alias("no_of_matches_played")
                ,sum(when(col("team_1") == col("winner"), 1).otherwise(0)).alias("no_of_matches_won")
                ,sum(when(col("team_1") != col("winner"), 1).otherwise(0)).alias("no_of_losses")
            )
            .select(
                col("team_1").alias("team_name")
                , col("no_of_matches_played")
                , col("no_of_matches_won")
                , col("no_of_losses")
            )
            .orderBy(col("no_of_matches_won").desc())
        )

display(result)

## Employees Earning More Than Their Manager

### Table: `emp`

| Column Name | Type |
| --- | --- |
| `emp_id` | `int` |
| `emp_name` | `varchar(10)` |
| `salary` | `int` |
| `manager_id` | `int` |

* Each employee has a `manager_id`.
* `manager_id` refers to `emp_id` within the same table (self-referencing relationship).

### Task
Write a SQL query to find employees whose salary is greater than their manager's salary.

This is a **self join** scenario — one column (`manager_id`) references another column (`emp_id`) of the same table.

### Example

**Input:** `emp`

| emp_id | emp_name | salary | manager_id |
| --- | --- | --- | --- |
| 1 | Ankit | 60000 | 10 |
| 2 | Rahul | 75000 | 10 |
| 3 | Amit | 50000 | 10 |
| 4 | Neha | 82000 | 11 |
| 5 | Priya | 90000 | 11 |
| 6 | Ravi | 45000 | 12 |
| 7 | Sunil | 65000 | 12 |
| 8 | Karan | 72000 | 13 |
| 9 | Vikas | 68000 | 13 |
| 10 | Rohit | 70000 | 15 |
| 11 | Mohit | 85000 | 15 |
| 12 | Suresh | 60000 | 16 |
| 13 | Deepak | 75000 | 16 |
| 14 | Pooja | 78000 | 16 |
| 15 | Aakash | 80000 | 18 |
| 16 | Nitin | 70000 | 18 |
| 17 | Ritika | 88000 | 18 |
| 18 | Sanjay | 90000 | null |
| 19 | Divya | 65000 | 18 |
| 20 | Manish | 95000 | 18 |

**Output:**

| emp_id | emp_name | manager_name | salary | manager_salary |
| --- | --- | --- | --- | --- |
| 2 | Rahul | Rohit | 75000 | 70000 |
| 5 | Priya | Mohit | 90000 | 85000 |
| 7 | Sunil | Suresh | 65000 | 60000 |
| 11 | Mohit | Aakash | 85000 | 80000 |
| 13 | Deepak | Nitin | 75000 | 70000 |
| 14 | Pooja | Nitin | 78000 | 70000 |
| 20 | Manish | Sanjay | 95000 | 90000 |

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

emp_schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("emp_name", StringType(), True),
    StructField("salary", IntegerType(), True),
    StructField("manager_id", IntegerType(), True)
])

emp_data = [
    (1, "Ankit", 60000, 10),
    (2, "Rahul", 75000, 10),
    (3, "Amit", 50000, 10),
    (4, "Neha", 82000, 11),
    (5, "Priya", 90000, 11),
    (6, "Ravi", 45000, 12),
    (7, "Sunil", 65000, 12),
    (8, "Karan", 72000, 13),
    (9, "Vikas", 68000, 13),
    (10, "Rohit", 70000, 15),
    (11, "Mohit", 85000, 15),
    (12, "Suresh", 60000, 16),
    (13, "Deepak", 75000, 16),
    (14, "Pooja", 78000, 16),
    (15, "Aakash", 80000, 18),
    (16, "Nitin", 70000, 18),
    (17, "Ritika", 88000, 18),
    (18, "Sanjay", 90000, None),
    (19, "Divya", 65000, 18),
    (20, "Manish", 95000, 18)
]

emp = spark.createDataFrame(emp_data, schema=emp_schema)
emp.createOrReplaceTempView("emp")

In [0]:
%sql
SELECT
    a.emp_id
    , a.emp_name
    , b.emp_name AS manager_name
    , a.salary
    , b.salary AS manager_salary
FROM
    emp a
JOIN
    emp b
ON
    a.manager_id = b.emp_id
WHERE
    a.salary > b.salary
ORDER BY
    a.emp_id;


In [0]:
a = emp.alias("a")
b = emp.alias("b")

result = (a
          .join(b, a["manager_id"] == b["emp_id"], "inner")
          .filter(a["salary"] > b["salary"])
          .select(
              a["emp_id"]
              , a["emp_name"]
              , b["emp_name"].alias("manager_name")
              , a["salary"]
              , b["salary"].alias("manager_salary")
          )
          .orderBy("emp_id")
        )

display(result)

## New vs Repeat Customers Per Day

### Table: `customer_orders`

| Column Name | Type |
| --- | --- |
| `order_id` | `integer` |
| `customer_id` | `integer` |
| `order_date` | `text` (YYYY-MM-DD) |
| `order_amount` | `integer` |

* Each row represents an order placed by a customer on a given date.
* Some customers are **new** (first-time buyers).
* Some customers are **repeat** buyers (have placed an order before).

### Task
For each day, find:

* Number of **new customers**
* Number of **repeat customers**

A customer is considered *new* on a given date if they have no prior orders on any earlier date.
A customer is considered *repeat* on a given date if they have at least one order on an earlier date.

### Example

**Input:** `customer_orders`

| order_id | customer_id | order_date | order_amount |
| --- | --- | --- | --- |
| 1 | 100 | 2023-01-01 | 500 |
| 2 | 200 | 2023-01-01 | 700 |
| 3 | 300 | 2023-01-01 | 400 |
| 4 | 100 | 2023-01-02 | 600 |
| 5 | 400 | 2023-01-02 | 800 |
| 6 | 500 | 2023-01-02 | 300 |
| 7 | 100 | 2023-01-03 | 900 |
| 8 | 400 | 2023-01-03 | 200 |
| 9 | 600 | 2023-01-03 | 1000 |

**Output:**

| order_date | new_customers | repeat_customers |
| --- | --- | --- |
| 2023-01-01 | 3 | 0 |
| 2023-01-02 | 2 | 1 |
| 2023-01-03 | 1 | 2 |

In [0]:
data = [
    (1, 100, "2023-01-01", 500),
    (2, 200, "2023-01-01", 700),
    (3, 300, "2023-01-01", 400),
    (4, 100, "2023-01-02", 600),
    (5, 400, "2023-01-02", 800),
    (6, 500, "2023-01-02", 300),
    (7, 100, "2023-01-03", 900),
    (8, 400, "2023-01-03", 200),
    (9, 600, "2023-01-03", 1000)
]

customer_orders = spark.createDataFrame(data, schema=["customer_id", "order_id", "order_date", "order_amount"])
customer_orders.createOrReplaceTempView("customer_orders")

In [0]:
%sql
SELECT
    order_date
    ,new_customers
    , repeat_customers
FROM
    customer_orders
